In [1]:
import numpy as np
from scipy.io import loadmat
from scipy.signal import savgol_filter  


def dataset_to_iddata(dataset, K, F):

    runs = dataset.runs
    # When using squeeze_me=True, single dimension structs become 1D arrays
    if not isinstance(runs, np.ndarray):
        runs = np.array([runs])

    R = runs.size

    # --- first pass: total number of samples ---
    Ni = np.zeros(R, dtype=int)
    for i in range(R):
        Ni[i] = runs[i].x_true.shape[0]  # length of this run
    Ntot = Ni.sum()

    # Preallocate
    data = {}
    data["t"]        = np.zeros(Ntot)
    data["q_meas"]   = np.zeros((2, Ntot))
    data["qd_filt"]  = np.zeros((2, Ntot))
    data["qdd_filt"] = np.zeros((2, Ntot))
    data["q"]        = np.zeros((2, Ntot))
    data["qd"]       = np.zeros((2, Ntot))
    data["qdd"]      = np.zeros((2, Ntot))
    data["tau_meas"] = np.zeros((2, Ntot))

    # assume constant dt across runs
    dt = runs[0].time[1] - runs[0].time[0]
    data["dt"] = float(dt)

    idx0 = 0  # starting index in the global arrays for each run
    for i in range(R):
        r = runs[i]
        ti = np.asarray(r.time).ravel()
        Ni_i = Ni[i]

        start = idx0
        end = idx0 + Ni_i  # Python slice is [start, end)

        # Fill raw (unfiltered) data
        for k in range(Ni_i):
            j = start + k  # global index
            xk = np.asarray(r.x_true[k, :]).ravel()   # [th1 th2 dth1 dth2]
            yk = np.asarray(r.y_meas[k, :]).ravel()   # [th1 th2 dth1 dth2]

            data["t"][j]            = ti[k]
            data["q_meas"][:, j]    = yk[0:2]         # measured positions
            data["q"][:, j]         = xk[0:2]         # true positions
            data["qd"][:, j]        = xk[2:4]         # true velocities
            data["qdd"][:, j]       = np.asarray(r.ddq[k, :]).ravel()
            data["tau_meas"][:, j]  = np.asarray(r.u_applied[k, :]).ravel()

        # ---- Filtering for this run ----
        # Position filtering with Savitzky–Golay (sgolayfilt)
        q_meas_run = data["q_meas"][:, start:end]  # shape (2, Ni_i)

        q_filt1 = savgol_filter(q_meas_run[0, :], window_length=F, polyorder=K, mode="interp")
        q_filt2 = savgol_filter(q_meas_run[1, :], window_length=F, polyorder=K, mode="interp")
        data["q_filt"] = data.get("q_filt", np.zeros((2, Ntot)))
        data["q_filt"][0, start:end] = q_filt1
        data["q_filt"][1, start:end] = q_filt2

        # Velocities from central differences of filtered positions
        q_filt_run = data["q_filt"][:, start:end]  # (2, Ni_i)

        # interior points
        for j in range(start + 1, end - 1):
            idx_prev = j - 1
            idx_next = j + 1
            data["qd_filt"][:, j] = (data["q_filt"][:, idx_next] -
                                     data["q_filt"][:, idx_prev]) / (2.0 * data["dt"])

        # boundaries (one-sided)
        data["qd_filt"][:, start] = (data["q_filt"][:, start + 1] -
                                     data["q_filt"][:, start]) / data["dt"]
        data["qd_filt"][:, end - 1] = (data["q_filt"][:, end - 1] -
                                       data["q_filt"][:, end - 2]) / data["dt"]

        # Filter velocities
        qd_run = data["qd_filt"][:, start:end]
        qd_filt1 = savgol_filter(qd_run[0, :], window_length=F, polyorder=K, mode="interp")
        qd_filt2 = savgol_filter(qd_run[1, :], window_length=F, polyorder=K, mode="interp")
        data["qd_filt"][0, start:end] = qd_filt1
        data["qd_filt"][1, start:end] = qd_filt2

        # Accelerations from filtered velocities
        qd_filt_run = data["qd_filt"][:, start:end]

        for j in range(start + 1, end - 1):
            idx_prev = j - 1
            idx_next = j + 1
            data["qdd_filt"][:, j] = (data["qd_filt"][:, idx_next] -
                                      data["qd_filt"][:, idx_prev]) / (2.0 * data["dt"])

        data["qdd_filt"][:, start] = (data["qd_filt"][:, start + 1] -
                                      data["qd_filt"][:, start]) / data["dt"]
        data["qdd_filt"][:, end - 1] = (data["qd_filt"][:, end - 1] -
                                        data["qd_filt"][:, end - 2]) / data["dt"]

        # Optional: smooth accelerations too (as in MATLAB)
        qdd_run = data["qdd_filt"][:, start:end]
        qdd_filt1 = savgol_filter(qdd_run[0, :], window_length=F, polyorder=K, mode="interp")
        qdd_filt2 = savgol_filter(qdd_run[1, :], window_length=F, polyorder=K, mode="interp")
        data["qdd_filt"][0, start:end] = qdd_filt1
        data["qdd_filt"][1, start:end] = qdd_filt2

        # move starting index for next run
        idx0 = end

    return data


if __name__ == "__main__":
    # Load the same .mat file as in MATLAB
    S = loadmat("planar_2dof_dataset_gaussian_noise.mat",
                squeeze_me=True, struct_as_record=False)
    dataset = S["dataset"]

    F = 251
    K = 2
    data = dataset_to_iddata(dataset, K, F)


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class DynamicsDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X.astype(np.float32)
        self.Y = Y.astype(np.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


class FFNN(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_sizes):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [24]:
import random

SEED = 42
BATCH_SIZE = 256
LR = 2e-3
EPOCHS = 500
HIDDEN_SIZES = [128, 64]
WEIGHT_DECAY = 1e-5
MODEL_OUT = "ffnn_model.pth"
DEVICE = "cuda" if __import__('torch').cuda.is_available() else "cpu"

set_seed(SEED)
# Build features and targets
# Use filtered positions/velocities/accelerations where available
q = data["q_filt"]    # (2, N)
qd = data["qd_filt"]  # (2, N)
qdd = data["qdd_filt"] # (2, N)
tau = data["tau_meas"] # (2, N)

# Stack into (N, 6) and (N, 2)
# Features: positions + velocities
X = np.vstack([q, qd, tau]).T  # shape (N,6)

# Targets: accelerations (predicted)
Y = qdd.T     # shape (N, 2)

# Remove NaNs or infs if any
mask = np.isfinite(X).all(axis=1) & np.isfinite(Y).all(axis=1)
X = X[mask]
Y = Y[mask]

# Simple normalization (mean/std) on input and target
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-9
Y_mean = Y.mean(axis=0)
Y_std = Y.std(axis=0) + 1e-9

Xn = (X - X_mean) / X_std
Yn = (Y - Y_mean) / Y_std

# Train/val split
N = Xn.shape[0]
idx = np.arange(N)
np.random.shuffle(idx)
split = int(0.8 * N)
train_idx = idx[:split]
val_idx = idx[split:]

train_ds = DynamicsDataset(Xn[train_idx], Yn[train_idx])
val_ds = DynamicsDataset(Xn[val_idx], Yn[val_idx])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# Model
model = FFNN(in_dim=6, out_dim=2, hidden_sizes=HIDDEN_SIZES)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

best_val = float('inf')
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred = model(xb)
            loss = criterion(pred, yb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    if val_loss < best_val:
        best_val = val_loss
        # Save state dict plus normalization
        save_dict = {
            'model_state_dict': model.state_dict(),
            'X_mean': X_mean, 'X_std': X_std,
            'Y_mean': Y_mean, 'Y_std': Y_std,
            'hidden_sizes': HIDDEN_SIZES
        }
        torch.save(save_dict, MODEL_OUT)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train loss {train_loss:.6f} | val loss {val_loss:.6f}")

print("Training finished. Best val loss:", best_val)
print("Saved model to:", MODEL_OUT)

# Quick evaluation on validation set: return denormalized RMSE per joint
checkpoint = torch.load(MODEL_OUT, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE).eval()
with torch.no_grad():
    Xv = torch.from_numpy(Xn[val_idx]).float().to(DEVICE)
    Yv = torch.from_numpy(Y[val_idx]).float().to(DEVICE)
    predn = model(Xv).cpu().numpy()

# denormalize
pred = predn * Y_std + Y_mean
true = Y[val_idx]
rmse = np.sqrt(np.mean((pred - true) ** 2, axis=0))
print(f"Validation RMSE per joint: {rmse}")



Epoch   1 | train loss 0.403933 | val loss 0.288687
Epoch  10 | train loss 0.022575 | val loss 0.021683
Epoch  20 | train loss 0.013583 | val loss 0.013799
Epoch  30 | train loss 0.011146 | val loss 0.010741
Epoch  40 | train loss 0.009460 | val loss 0.011479
Epoch  50 | train loss 0.008597 | val loss 0.009356
Epoch  60 | train loss 0.007728 | val loss 0.006981
Epoch  70 | train loss 0.007175 | val loss 0.006495
Epoch  80 | train loss 0.006641 | val loss 0.006096
Epoch  90 | train loss 0.006330 | val loss 0.005730
Epoch 100 | train loss 0.006316 | val loss 0.006708
Epoch 110 | train loss 0.005725 | val loss 0.005976
Epoch 120 | train loss 0.006128 | val loss 0.007925
Epoch 130 | train loss 0.005552 | val loss 0.005822
Epoch 140 | train loss 0.005321 | val loss 0.005198
Epoch 150 | train loss 0.005444 | val loss 0.007302
Epoch 160 | train loss 0.005064 | val loss 0.004030
Epoch 170 | train loss 0.004935 | val loss 0.004608
Epoch 180 | train loss 0.005061 | val loss 0.004721
Epoch 190 | 

In [25]:
# Example input shape: [batch, features] = [1,6]
x_example = torch.zeros(1,6)  # q1,q2,qd1,qd2,tau1,tau2

torch.onnx.export(model, x_example, "ffnn_model.onnx",
                  input_names=["x_in"],
                  output_names=["qdd_out"],
                  dynamic_axes={"x_in": {0: "batch"}, "qdd_out": {0: "batch"}},
                  opset_version=17)

/var/folders/b_/w2_d_pr10y73nq82f53t_m4c0000gn/T/ipykernel_5067/3652758587.py:4: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(model, x_example, "ffnn_model.onnx",
W1120 13:28:04.491000 5067 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W1120 13:28:04.751000 5067 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms


[torch.onnx] Obtain model graph for `FFNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `FFNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 17},
            producer_name='pytorch',
            producer_version='2.9.1',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"x_in"<FLOAT,[s77,6]>
            ),
            outputs=(
                %"qdd_out"<FLOAT,[s77,2]>
            ),
            initializers=(
                %"net.0.weight"<FLOAT,[128,6]>{TorchTensor(...)},
                %"net.0.bias"<FLOAT,[128]>{TorchTensor(...)},
                %"net.2.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"net.4.weight"<FLOAT,[2,64]>{TorchTensor(...)},
                %"net.4.bias"<FLOAT,[2]>{TorchTensor<FLOAT,[2]>(Parameter containing: tensor([-0.0868, -0.1809], requires_grad=True), name='net.4.bias')},
                %"net.2.weight"<FLOAT,[64,128]>{TorchTensor(...)}
            ),
        ) {
            0 |  # node_linear
   